<a href="https://colab.research.google.com/github/PromyotKatarat/AI_agent_engineering/blob/Phase-13-%C2%B7-Tools-%26-Protocols/The_Tool_Interface_%E2%80%94_Why_Agents_Need_Structured_I_O.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. ดึงโปรเจกต์มาทั้งหมดก่อน
!git clone https://github.com/rohitg00/ai-engineering-from-scratch.git

# 2. ย้ายพาธเข้าไปที่โฟลเดอร์หลักของ Agent Engineering
%cd ai-engineering-from-scratch/phases/14-agent-engineering/

# 3. ลองเช็กดูซิว่ามีโฟลเดอร์บทเรียนอะไรให้เล่นบ้าง
!ls -l

Cloning into 'ai-engineering-from-scratch'...
remote: Enumerating objects: 15928, done.
remote: Counting objects: 100% (7494/7494), done.
remote: Compressing objects: 100% (3252/3252), done.
remote: Total 15928 (delta 4613), reused 4242 (delta 4242), pack-reused 8434 (from 2)
Receiving objects: 100% (15928/15928), 8.74 MiB | 13.94 MiB/s, done.
Resolving deltas: 100% (7337/7337), done.
/content/ai-engineering-from-scratch/phases/14-agent-engineering
total 172
drwxr-xr-x 7 root root 4096 Jun  3 07:15 01-the-agent-loop
drwxr-xr-x 7 root root 4096 Jun  3 07:15 02-rewoo-plan-and-execute
drwxr-xr-x 7 root root 4096 Jun  3 07:15 03-reflexion-verbal-rl
drwxr-xr-x 7 root root 4096 Jun  3 07:15 04-tree-of-thoughts-lats
drwxr-xr-x 7 root root 4096 Jun  3 07:15 05-self-refine-and-critic
drwxr-xr-x 7 root root 4096 Jun  3 07:15 06-tool-use-and-function-calling
drwxr-xr-x 7 root root 4096 Jun  3 07:15 07-memory-virtual-context-memgpt
drwxr-xr-x 7 root root 4096 Jun  3 07:15 08-memory-blocks-sleep-ti

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Callable

@dataclass
class ToolCall:
    name: str
    args: dict[str, Any]

@dataclass
class Turn:
    kind: str
    content: str
    tool_call: ToolCall | None = None
    observation: str | None = None

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        self._tools[name] = fn

    def names(self) -> list[str]:
        return sorted(self._tools)

    def dispatch(self, call: ToolCall) -> str:
        fn = self._tools.get(call.name)
        if fn is None:
            return f"error: unknown tool {call.name!r}"
        try:
            return fn(**call.args)
        except TypeError as e:
            return f"error: bad args for {call.name}: {e}"
        except Exception as e:
            return f"error: {type(e).__name__}: {e}"

def calculator(expr: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "error: illegal character in expr"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {type(e).__name__}: {e}"

class KVStore:
    def __init__(self) -> None:
        self._store: dict[str, str] = {}

    def get(self, key: str) -> str:
        return self._store.get(key, f"missing:{key}")

    def set(self, key: str, value: str) -> str:
        self._store[key] = value
        return f"stored {key}"

class ToyLLM:
    def __init__(self, script: list[dict[str, Any]]) -> None:
        self.script = script
        self.cursor = 0

    def respond(self, history: list[Turn]) -> dict[str, Any]:
        if self.cursor >= len(self.script):
            return {"kind": "finish", "content": "no more actions"}
        entry = self.script[self.cursor]
        self.cursor += 1
        return entry

# ======================================================================
# 🌟 คลาส AgentLoop ที่แก้ไขเพื่อทำแลปข้อ 1 และ ข้อ 2 เรียบร้อยแล้ว 🌟
# ======================================================================
@dataclass
class AgentLoop:
    llm: ToyLLM
    tools: ToolRegistry
    max_turns: int = 12
    max_tool_calls_per_turn: int = 1  # 🛠️ ข้อ 1: ตั้งค่าจำกัดการเรียกเครื่องมือสูงสุด "ต่อหนึ่งสเต็ปการคิด"
    history: list[Turn] = field(default_factory=list)

    def run(self, user_message: str) -> str:
        self.history.append(Turn(kind="user", content=user_message))

        for step in range(self.max_turns):
            # 🛠️ ข้อ 1 (แก้ไขอาการนับข้ามเลน): รีเซ็ตตัวนับการใช้เครื่องมือกลับเป็น 0 ทุกครั้งที่ขึ้นสเต็ป (Turn) ใหม่
            tool_calls_this_turn = 0

            reply = self.llm.respond(self.history)

            # --- 🛠️ แลปข้อ 2 (Explicit Finish) ---
            if reply["kind"] == "finish":
                self.history.append(Turn(kind="final", content=reply["content"]))
                return reply["content"]

            # --- 🛠️ แลปข้อ 2 (no_tool_calls -> done stop path) ---
            # ดักจับกรณีที่ AI ส่งผลลัพธ์มาแต่ไม่ได้ระบุคำสั่งเรียกเครื่องมือ (Action)
            if "action" not in reply or reply["action"] is None:
                final_content = reply.get("thought", "Done (No tools called)")
                self.history.append(Turn(kind="final", content=f"[Auto-Done]: {final_content}"))
                return final_content
            # -------------------------------------------------------------

            thought = reply.get("thought", "")
            self.history.append(Turn(kind="thought", content=thought))

            call = ToolCall(name=reply["action"], args=reply.get("args", {}))

            # 🛠️ ข้อ 1: ตรวจสอบโควตาการเรียกเครื่องมือในรอบนั้นๆ
            if tool_calls_this_turn >= self.max_tool_calls_per_turn:
                observation = f"error: Max tool calls per turn ({self.max_tool_calls_per_turn}) exceeded. Tool {call.name!r} skipped."
                print(f"⚠️ [ระบบเบรกมือ]: {observation}")
            else:
                observation = self.tools.dispatch(call)
                tool_calls_this_turn += 1  # นับเพิ่มขึ้นเมื่อยอมให้รันสำเร็จจริง

            self.history.append(
                Turn(kind="action", content=call.name, tool_call=call, observation=observation)
            )

        self.history.append(Turn(kind="final", content="budget exhausted"))
        return "budget exhausted"

def pretty_trace(history: list[Turn]) -> None:
    for i, turn in enumerate(history):
        tag = f"[{i:02d} {turn.kind:>7}]"
        if turn.kind == "user":
            print(f"{tag} {turn.content}")
        elif turn.kind == "thought":
            print(f"{tag} {turn.content}")
        elif turn.kind == "action":
            call = turn.tool_call
            assert call is not None
            print(f"{tag} {call.name}({call.args}) -> {turn.observation}")
        elif turn.kind == "final":
            print(f"{tag} {turn.content}")

def build_demo_agent() -> AgentLoop:
    tools = ToolRegistry()
    tools.register("calculator", calculator)
    kv = KVStore()
    tools.register("kv_get", kv.get)
    tools.register("kv_set", kv.set)

    script: list[dict[str, Any]] = [
        {"kind": "action", "thought": "store the base price", "action": "kv_set", "args": {"key": "base", "value": "120"}},
        {"kind": "action", "thought": "compute 15% tax", "action": "calculator", "args": {"expr": "120 * 0.15"}},
        {"kind": "action", "thought": "store the tax", "action": "kv_set", "args": {"key": "tax", "value": "18.0"}},
        {"kind": "action", "thought": "compute total", "action": "calculator", "args": {"expr": "120 + 18.0"}},
        {"kind": "action", "thought": "confirm stored values", "action": "kv_get", "args": {"key": "base"}},
        {"kind": "finish", "content": "the total including 15% tax is 138.0"},
    ]
    return AgentLoop(llm=ToyLLM(script), tools=tools, max_turns=10)

def main() -> None:
    print("=" * 70)
    print("TOY REACT LOOP — Phase 14, Lesson 01")
    print("=" * 70)
    agent = build_demo_agent()
    final = agent.run("What is 120 plus 15% tax, stored in kv?")
    print()
    pretty_trace(agent.history)
    print()
    print(f"final answer: {final}")
    print(f"turns used:   {len([t for t in agent.history if t.kind == 'action'])}")
    print(f"tools used:   {agent.tools.names()}")

if __name__ == "__main__":
    main()

TOY REACT LOOP — Phase 14, Lesson 01

[00    user] What is 120 plus 15% tax, stored in kv?
[01 thought] store the base price
[02  action] kv_set({'key': 'base', 'value': '120'}) -> stored base
[03 thought] compute 15% tax
[04  action] calculator({'expr': '120 * 0.15'}) -> 18.0
[05 thought] store the tax
[06  action] kv_set({'key': 'tax', 'value': '18.0'}) -> stored tax
[07 thought] compute total
[08  action] calculator({'expr': '120 + 18.0'}) -> 138.0
[09 thought] confirm stored values
[10  action] kv_get({'key': 'base'}) -> 120
[11   final] the total including 15% tax is 138.0

final answer: the total including 15% tax is 138.0
turns used:   5
tools used:   ['calculator', 'kv_get', 'kv_set']


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Callable

@dataclass
class ToolCall:
    name: str
    args: dict[str, Any]

@dataclass
class Turn:
    kind: str
    content: str
    tool_call: ToolCall | None = None
    observation: str | None = None

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        self._tools[name] = fn

    def names(self) -> list[str]:
        return sorted(self._tools)

    def dispatch(self, call: ToolCall) -> str:
        fn = self._tools.get(call.name)
        if fn is None:
            return f"error: unknown tool {call.name!r}"
        try:
            return fn(**call.args)
        except TypeError as e:
            return f"error: bad args for {call.name}: {e}"
        except Exception as e:
            return f"error: {type(e).__name__}: {e}"

def calculator(expr: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "error: illegal character in expr"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {type(e).__name__}: {e}"

class KVStore:
    def __init__(self) -> None:
        self._store: dict[str, str] = {}

    def get(self, key: str) -> str:
        return self._store.get(key, f"missing:{key}")

    def set(self, key: str, value: str) -> str:
        self._store[key] = value
        return f"stored {key}"

class ToyLLM:
    def __init__(self, script: list[dict[str, Any]]) -> None:
        self.script = script
        self.cursor = 0

    def respond(self, history: list[Turn]) -> dict[str, Any]:
        if self.cursor >= len(self.script):
            return {"kind": "finish", "content": "no more actions"}
        entry = self.script[self.cursor]
        self.cursor += 1
        return entry

# ======================================================================
# 🌟 คลาส AgentLoop ที่ฝังระบบตัดจบอัตโนมัติ (no_tool_calls -> done)
# ======================================================================
@dataclass
class AgentLoop:
    llm: ToyLLM
    tools: ToolRegistry
    max_turns: int = 12
    history: list[Turn] = field(default_factory=list)

    def run(self, user_message: str) -> str:
        self.history.append(Turn(kind="user", content=user_message))

        for step in range(self.max_turns):
            reply = self.llm.respond(self.history)

            # ทางเลือกที่ A: จบงานแบบเป็นทางการ (Explicit Finish)
            if reply["kind"] == "finish":
                self.history.append(Turn(kind="final", content=reply["content"]))
                return reply["content"]

            # --- 🛠️ ท่อนโค้ดดักจบงานอัตโนมัติ (no_tool_calls -> done) ---
            # เช็กว่ารอบนี้ AI ส่งผลลัพธ์มาแต่ไม่ได้ระบุ Action สั่งรันเครื่องมือใช่ไหม?
            if "action" not in reply or reply["action"] is None:
                final_content = reply.get("thought", "Done (No tools called)")

                # สั่งดีดตัดจบปิดงานทันที!
                self.history.append(Turn(kind="final", content=f"[Auto-Done]: {final_content}"))
                return final_content
            # -----------------------------------------------------------

            thought = reply.get("thought", "")
            self.history.append(Turn(kind="thought", content=thought))

            call = ToolCall(name=reply["action"], args=reply.get("args", {}))
            observation = self.tools.dispatch(call)

            self.history.append(
                Turn(kind="action", content=call.name, tool_call=call, observation=observation)
            )

        self.history.append(Turn(kind="final", content="budget exhausted"))
        return "budget exhausted"

def pretty_trace(history: list[Turn]) -> None:
    for i, turn in enumerate(history):
        tag = f"[{i:02d} {turn.kind:>7}]"
        if turn.kind == "user":
            print(f"{tag} {turn.content}")
        elif turn.kind == "thought":
            print(f"{tag} {turn.content}")
        elif turn.kind == "action":
            call = turn.tool_call
            assert call is not None
            print(f"{tag} {call.name}({call.args}) -> {turn.observation}")
        elif turn.kind == "final":
            print(f"{tag} {turn.content}")

def build_demo_agent() -> AgentLoop:
    tools = ToolRegistry()
    tools.register("calculator", calculator)
    kv = KVStore()
    tools.register("kv_get", kv.get)
    tools.register("kv_set", kv.set)

    # ⚠️ แกล้งโมเดลในสเต็ปที่ 3 ลบ "action" ออก ให้เหลือแต่ความคิดลอยๆ
    script: list[dict[str, Any]] = [
        {"kind": "action", "thought": "store the base price", "action": "kv_set", "args": {"key": "base", "value": "120"}},
        {"kind": "action", "thought": "compute 15% tax", "action": "calculator", "args": {"expr": "120 * 0.15"}},

        # 🔥 จุดที่โดนแกล้ง: ไม่มีคำสั่ง action เรียกเครื่องมือ (มีแต่ความคิดบ่นพึมพำ)
        {"kind": "action", "thought": "เอ... ภาษีได้ 18.0 แล้ว ต้องทำอะไรต่อนะ... ลืมคิดแป๊บ"},

        {"kind": "action", "thought": "compute total", "action": "calculator", "args": {"expr": "120 + 18.0"}},
        {"kind": "action", "thought": "confirm stored values", "action": "kv_get", "args": {"key": "base"}},
        {"kind": "finish", "content": "the total including 15% tax is 138.0"},
    ]
    return AgentLoop(llm=ToyLLM(script), tools=tools, max_turns=10)

def main() -> None:
    print("=" * 70)
    print("TESTING: no_tool_calls -> done stop path")
    print("=" * 70)
    agent = build_demo_agent()
    final = agent.run("What is 120 plus 15% tax, stored in kv?")
    print()
    pretty_trace(agent.history)
    print()
    print(f"final answer: {final}")

if __name__ == "__main__":
    main()

TESTING: no_tool_calls -> done stop path

[00    user] What is 120 plus 15% tax, stored in kv?
[01 thought] store the base price
[02  action] kv_set({'key': 'base', 'value': '120'}) -> stored base
[03 thought] compute 15% tax
[04  action] calculator({'expr': '120 * 0.15'}) -> 18.0
[05   final] [Auto-Done]: เอ... ภาษีได้ 18.0 แล้ว ต้องทำอะไรต่อนะ... ลืมคิดแป๊บ

final answer: เอ... ภาษีได้ 18.0 แล้ว ต้องทำอะไรต่อนะ... ลืมคิดแป๊บ


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Callable

@dataclass
class ToolCall:
    name: str
    args: dict[str, Any]

@dataclass
class Turn:
    kind: str
    content: str
    tool_call: ToolCall | None = None
    observation: str | None = None

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        self._tools[name] = fn

    def dispatch(self, call: ToolCall) -> str:
        fn = self._tools.get(call.name)
        if fn is None:
            return f"error: unknown tool {call.name!r}"
        try:
            return fn(**call.args)
        except Exception as e:
            return f"error: {e}"

# ======================================================================
# 🧠 คลาส AgentLoop พิเศษ: รวมทั้ง 2 ระบบไว้ให้เปิด-ปิดสวิตช์ทดสอบ
# ======================================================================
@dataclass
class AgentLoop:
    llm: ToyLLM
    tools: ToolRegistry
    max_turns: int = 12
    use_no_tool_shortcut: bool = False  # 🌟 สวิตช์เปิด-ปิดระบบทางลัด no_tool_calls
    history: list[Turn] = field(default_factory=list)

    def run(self, user_message: str) -> str:
        self.history = []  # รีเซ็ตประวัติใหม่ทุกครั้งที่รัน
        self.history.append(Turn(kind="user", content=user_message))

        for step in range(self.max_turns):
            reply = self.llm.respond(self.history)

            # [แนวทางที่ 1]: จบงานแบบเป็นทางการ (Explicit Finish) -> ระบบมาตรฐาน
            if reply["kind"] == "finish":
                self.history.append(Turn(kind="final", content=reply["content"]))
                return reply["content"]

            # [แนวทางที่ 2]: ทางลัดเจ้าปัญหา (no_tool_calls -> done)
            # ถ้าระบบเปิดสวิตช์นี้ไว้ และรอบนั้น AI ไม่ยอมสั่งเครื่องมือ (Action)
            if self.use_no_tool_shortcut and ("action" not in reply or reply["action"] is None):
                final_content = reply.get("thought", "Done")
                self.history.append(Turn(kind="final", content=f"[Auto-Done ทางลัด]: {final_content}"))
                return final_content

            # ตรรกะประมวลผล ReAct ปกติ
            thought = reply.get("thought", "")
            self.history.append(Turn(kind="thought", content=thought))

            if "action" in reply and reply["action"] is not None:
                call = ToolCall(name=reply["action"], args=reply.get("args", {}))
                observation = self.tools.dispatch(call)
            else:
                # ถ้าไม่เปิดทางลัด แต่โมเดลไม่มี action ส่งมา ระบบจะป้อน Error เตือนสติกลับไป
                observation = "error: System expected an Action, but you provided none. Please continue the task."

            self.history.append(
                Turn(kind="action", content=reply.get("action", "unknown"),
                     tool_call=ToolCall(name=reply.get("action", "none"), args={}), observation=observation)
            )

        return "budget exhausted"

class ToyLLM:
    def __init__(self, script: list[dict[str, Any]]) -> None:
        self.script = script
        self.cursor = 0

    def respond(self, history: list[Turn]) -> dict[str, Any]:
        if self.cursor >= len(self.script):
            return {"kind": "finish", "content": "no more actions"}
        entry = self.script[self.cursor]
        self.cursor += 1
        return entry

def pretty_trace(history: list[Turn]) -> None:
    for i, turn in enumerate(history):
        tag = f"[{i:02d} {turn.kind:>7}]"
        if turn.kind in ["user", "thought", "final"]:
            print(f"{tag} {turn.content}")
        elif turn.kind == "action":
            print(f"{tag} {turn.content} -> {turn.observation}")

# --- เตรียมระบบจำลอง ---
tools = ToolRegistry()
tools.register("calculator", lambda expr: str(eval(expr)))
tools.register("kv_set", lambda key, value: f"stored {key}")

# ⚠️ แกล้งโมเดลในสเต็ปที่ 3: ให้คิดในใจลอยๆ แต่ลบคำสั่งเครื่องมือ (action) ออกไป
script_with_bug = [
    {"kind": "action", "thought": "บันทึกราคาตั้งต้น", "action": "kv_set", "args": {"key": "base", "value": "120"}},
    {"kind": "action", "thought": "คำนวณภาษี 15%", "action": "calculator", "args": {"expr": "120 * 0.15"}},

    # 🔥 สเต็ปเจ้าปัญหา: โมเดลเงียบไป ดื้อๆ คิดในใจลอยๆ แต่ลืมสั่ง Action รันเครื่องมือ
    {"kind": "action", "thought": "เอ... ภาษีได้ 18 บาทแล้ว ต้องทำอะไรต่อหว่า... ลืมแป๊บ"},

    {"kind": "action", "thought": "คิดราคารวม", "action": "calculator", "args": {"expr": "120 + 18"}},
    {"kind": "finish", "content": "ราคารวมสุทธิคือ 138.0 บาทจ้า"}
]

# ======================================================================
# 🚀 รันเปรียบเทียบผลลัพธ์
# ======================================================================

print("=" * 80)
print("เคสที่ 1: ใช้ระบบทางลัด [no_tool_calls -> done] (สวิตช์เปิด)")
print("=" * 80)
llm1 = ToyLLM(list(script_with_bug))
agent_shortcut = AgentLoop(llm=llm1, tools=tools, use_no_tool_shortcut=True)
agent_shortcut.run("คำนวณราคาให้หน่อย")
pretty_trace(agent_shortcut.history)

print("\n" + "=" * 80)
print("เคสที่ 2: ใช้ระบบมาตรฐาน [Explicit Finish] (สวิตช์ปิด)")
print("=" * 80)
llm2 = ToyLLM(list(script_with_bug))
agent_explicit = AgentLoop(llm=llm2, tools=tools, use_no_tool_shortcut=False)
agent_explicit.run("คำนวณราคาให้หน่อย")
pretty_trace(agent_explicit.history)

เคสที่ 1: ใช้ระบบทางลัด [no_tool_calls -> done] (สวิตช์เปิด)
[00    user] คำนวณราคาให้หน่อย
[01 thought] บันทึกราคาตั้งต้น
[02  action] kv_set -> stored base
[03 thought] คำนวณภาษี 15%
[04  action] calculator -> 18.0
[05   final] [Auto-Done ทางลัด]: เอ... ภาษีได้ 18 บาทแล้ว ต้องทำอะไรต่อหว่า... ลืมแป๊บ

เคสที่ 2: ใช้ระบบมาตรฐาน [Explicit Finish] (สวิตช์ปิด)
[00    user] คำนวณราคาให้หน่อย
[01 thought] บันทึกราคาตั้งต้น
[02  action] kv_set -> stored base
[03 thought] คำนวณภาษี 15%
[04  action] calculator -> 18.0
[05 thought] เอ... ภาษีได้ 18 บาทแล้ว ต้องทำอะไรต่อหว่า... ลืมแป๊บ
[06  action] unknown -> error: System expected an Action, but you provided none. Please continue the task.
[07 thought] คิดราคารวม
[08  action] calculator -> 138
[09   final] ราคารวมสุทธิคือ 138.0 บาทจ้า


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Callable

@dataclass
class ToolCall:
    name: str
    args: dict[str, Any]

@dataclass
class Turn:
    kind: str
    content: str
    tool_call: ToolCall | None = None
    observation: str | None = None

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        self._tools[name] = fn

    def dispatch(self, call: ToolCall) -> str:
        fn = self._tools.get(call.name)
        if fn is None:
            return f"error: unknown tool {call.name!r}"
        try:
            # ใช้สัญกรณ์ตรวจสอบความถูกต้องเบื้องต้น
            return fn(**call.args)
        except TypeError as e:
            # 🌟 จับอาการ Argument ผิดพลาดพาสเกลพัง (Malformed Arguments)
            return f"error: malformed arguments for tool {call.name!r}: {e}"
        except Exception as e:
            return f"error: {type(e).__name__}: {e}"

def calculator(expr: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "error: illegal character in expr"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {type(e).__name__}: {e}"

class KVStore:
    def __init__(self) -> None:
        self._store: dict[str, str] = {}

    def get(self, key: str) -> str:
        return self._store.get(key, f"missing:{key}")

    def set(self, key: str, value: str) -> str:
        # บังคับโครงสร้างข้อมูลจำลอง: สมมติว่า key ต้องเป็น string เสมอ
        if not isinstance(key, str):
            raise TypeError("key must be a string")
        self._store[key] = value
        return f"stored {key}"

class ToyLLM:
    def __init__(self, script: list[dict[str, Any]]) -> None:
        self.script = script
        self.cursor = 0

    def respond(self, history: list[Turn]) -> dict[str, Any]:
        if self.cursor >= len(self.script):
            return {"kind": "finish", "content": "no more actions"}
        entry = self.script[self.cursor]
        self.cursor += 1
        return entry

# ======================================================================
# 🌟 คลาส AgentLoop ที่รองรับระบบกู้ชีพตนเอง (CRITIC Recovery Loop)
# ======================================================================
@dataclass
class AgentLoop:
    llm: ToyLLM
    tools: ToolRegistry
    max_turns: int = 12
    history: list[Turn] = field(default_factory=list)

    def run(self, user_message: str) -> str:
        self.history.append(Turn(kind="user", content=user_message))

        for step in range(self.max_turns):
            reply = self.llm.respond(self.history)

            if reply["kind"] == "finish":
                self.history.append(Turn(kind="final", content=reply["content"]))
                return reply["content"]

            thought = reply.get("thought", "")
            self.history.append(Turn(kind="thought", content=thought))

            # ดึงโครงสร้างคำสั่ง
            call = ToolCall(name=reply.get("action", "none"), args=reply.get("args", {}))

            # 🔥 ประมวลผลรันเครื่องมือ: ถ้าระบบพัง ตัว dispatch จะส่งข้อความ Error กลับมาแทนที่จะดับเครื่อง
            observation = self.tools.dispatch(call)

            self.history.append(
                Turn(kind="action", content=call.name, tool_call=call, observation=observation)
            )

        return "budget exhausted"

def pretty_trace(history: list[Turn]) -> None:
    print("-" * 80)
    for i, turn in enumerate(history):
        tag = f"[{i:02d} {turn.kind:>7}]"
        if turn.kind in ["user", "thought", "final"]:
            print(f"{tag} {turn.content}")
        elif turn.kind == "action":
            call = turn.tool_call
            assert call is not None
            print(f"{tag} {call.name}({call.args}) -> {turn.observation}")
    print("-" * 80)

def build_critic_demo_agent() -> AgentLoop:
    tools = ToolRegistry()
    tools.register("calculator", calculator)
    kv = KVStore()
    tools.register("kv_set", kv.set)

    # 🔄 จำลองบทละครที่มีการพ่น Argument พัง และมีการซ่อมแซมตัวเอง (CRITIC Style)
    critic_script = [
        # สเต็ป 1: รันคำนวณปกติ
        {"kind": "action", "thought": "คำนวณค่าของ 150 * 0.10", "action": "calculator", "args": {"expr": "150 * 0.10"}},

        # 🔥 สเต็ป 2 (Malformed Trigger): ส่งอาร์กิวเมนต์ผิดฟอร์แมต จงใจส่งชื่อ 'expr' แทนที่จะเป็น 'key' และ 'value'
        {"kind": "action", "thought": "เอาผลลัพธ์ 15 ไปบันทึกในคลัง", "action": "kv_set", "args": {"expr": "tax_value = 15"}},

        # 🌟 สเต็ป 3 (CRITIC Recovery): โมเดลอ่านเจอข้อความเตือนเออเร่อในอดีต จึงดึงสติกลับมาและส่งคำสั่งที่แก้ไขถูกต้อง
        {"kind": "action", "thought": "โอ๊ะ! ระบบแจ้งว่าอาร์กิวเมนต์ผิดพลาด ฉันต้องเปลี่ยนไปใช้ key และ value ให้ถูกต้อง", "action": "kv_set", "args": {"key": "tax", "value": "15"}},

        {"kind": "finish", "content": "ทำการแก้ไขข้อมูลและบันทึกค่าลงคลังสำเร็จเรียบร้อยแล้ว!"}
    ]
    return AgentLoop(llm=ToyLLM(critic_script), tools=tools, max_turns=10)

def main() -> None:
    print("=" * 80)
    print("RUNNING: 2026 CRITIC-Style Malformed Argument Recovery")
    print("=" * 80)
    agent = build_critic_demo_agent()
    final = agent.run("ช่วยประมวลผลระบบข้อมูลให้หน่อย")
    pretty_trace(agent.history)
    print(f"คำตอบสุดท้ายจากระบบ: {final}")

if __name__ == "__main__":
    main()

RUNNING: 2026 CRITIC-Style Malformed Argument Recovery
--------------------------------------------------------------------------------
[00    user] ช่วยประมวลผลระบบข้อมูลให้หน่อย
[01 thought] คำนวณค่าของ 150 * 0.10
[02  action] calculator({'expr': '150 * 0.10'}) -> 15.0
[03 thought] เอาผลลัพธ์ 15 ไปบันทึกในคลัง
[04  action] kv_set({'expr': 'tax_value = 15'}) -> error: malformed arguments for tool 'kv_set': KVStore.set() got an unexpected keyword argument 'expr'
[05 thought] โอ๊ะ! ระบบแจ้งว่าอาร์กิวเมนต์ผิดพลาด ฉันต้องเปลี่ยนไปใช้ key และ value ให้ถูกต้อง
[06  action] kv_set({'key': 'tax', 'value': '15'}) -> stored tax
[07   final] ทำการแก้ไขข้อมูลและบันทึกค่าลงคลังสำเร็จเรียบร้อยแล้ว!
--------------------------------------------------------------------------------
คำตอบสุดท้ายจากระบบ: ทำการแก้ไขข้อมูลและบันทึกค่าลงคลังสำเร็จเรียบร้อยแล้ว!


In [ ]:
pip install requests

In [ ]:
# 0. ติดตั้ง zstd ซึ่งเป็น dependency ที่ Ollama ต้องการ
!sudo apt-get install zstd -y

# 1. สั่งดาวน์โหลดและติดตั้ง Ollama ลงในเซิร์ฟเวอร์ของ Google Colab ดื้อๆ เลย
!curl -fsSL https://ollama.com/install.sh | sh

# 2. เปิดระบบ Ollama ให้ทำงานเบื้องหลัง (Background Process)
!nohup ollama serve > ollama.log 2>&1 &

# 3. สั่งโหลดโมเดล qwen2.5 เข้ามาใน Colab (รอหลอดโหลดวิ่งจนครบ 100%)
!ollama pull qwen2.5

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,095 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 118242 files and directories currently 

In [ ]:
from __future__ import annotations
import json
import requests
from dataclasses import dataclass, field
from typing import Any, Callable

@dataclass
class ToolCall:
    name: str
    args: dict[str, Any]

@dataclass
class Turn:
    role: str       # 'user', 'assistant', 'tool'
    content: str | None = None
    reasoning: str | None = None  # ช่องเก็บกระบวนการคิดแยก (Reasoning Channel)
    tool_calls: list[ToolCall] = field(default_factory=list)

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}
        self._specs: dict[str, dict[str, Any]] = {}

    def register(self, name: str, fn: Callable[..., str], description: str, parameters: dict[str, Any]) -> None:
        self._tools[name] = fn
        self._specs[name] = {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": parameters
            }
        }

    def get_tools_spec(self) -> list[dict[str, Any]]:
        return list(self._specs.values())

    def dispatch(self, name: str, args: dict[str, Any]) -> str:
        fn = self._tools.get(name)
        if fn is None:
            return f"error: unknown tool {name!r}"
        try:
            return fn(**args)
        except Exception as e:
            return f"error: {e}"

# --- เครื่องมือเสริมพลังให้ AI ---
def calculator(expr: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "error: illegal character"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

# ======================================================================
# 🌟 คลาส LocalOllamaAgent: คุมลูป ReAct ดึงสมองจาก Ollama ในเครื่อง
# ======================================================================
class OllamaAgentLoop:
    def __init__(self, tools: ToolRegistry, model: str = "qwen2.5", max_turns: int = 10):
        self.ollama_url = "http://localhost:11434/api/chat" # พอร์ตมาตรฐาน Ollama
        self.tools = tools
        self.model = model
        self.max_turns = max_turns
        self.history: list[Turn] = []

    def run(self, user_message: str) -> str:
        self.history.append(Turn(role="user", content=user_message))

        for step in range(self.max_turns):
            # 1. แปลงประวัติเป็นรูปแบบที่ Ollama API ต้องการ
            api_messages = []
            for turn in self.history:
                msg = {"role": turn.role}
                if turn.content is not None:
                    msg["content"] = turn.content

                # แนบประวัติการเรียกเครื่องมือ (ถ้ามี)
                if turn.role == "assistant" and turn.tool_calls:
                    msg["tool_calls"] = [
                        {
                            "function": {"name": tc.name, "arguments": tc.args}
                        } for tc in turn.tool_calls
                    ]
                api_messages.append(msg)

            # 2. ยิงเรียก Ollama (Local API)
            payload = {
                "model": self.model,
                "messages": api_messages,
                "stream": False,  # ปิด stream เพื่อเอาข้อความเต็มๆ มาประมวลผล
                "tools": self.tools.get_tools_spec() # ส่งปลั๊กอินเครื่องมือให้ AI รู้จัก
            }

            try:
                response = requests.post(self.ollama_url, json=payload)
                response.raise_for_status()
                res_json = response.json()
            except Exception as e:
                return f"🚨 เชื่อมต่อ Ollama ไม่สำเร็จ: กรุณาเปิดโปรแกรม Ollama ก่อนรันโค้ด ({e})"

            message = res_json.get("message", {})
            content = message.get("content", "")

            # 🧠 ดึงความคิดแยกท่อ (Ollama บางรุ่นจะส่งวิธีคิดมาในส่วนประกอบ หรือเราสกัดจากเนื้อหา)
            # เพื่อความแม่นยำ ลูปนี้จะตรวจเช็กโครงสร้าง Tool Calls อย่างเป็นทางการ
            api_tool_calls = message.get("tool_calls", [])
            current_tool_calls = []

            for tc in api_tool_calls:
                func = tc.get("function", {})
                current_tool_calls.append(
                    ToolCall(name=func.get("name"), args=func.get("arguments", {}))
                )

            # บันทึกสถานะลงระบบประวัติ
            self.history.append(Turn(
                role="assistant",
                content=content if content else None,
                reasoning="วิเคราะห์บริบทคำสั่งเพื่อเลือกเครื่องมือ..." if current_tool_calls else "ประมวลผลสรุปคำตอบสุดท้าย",
                tool_calls=current_tool_calls
            ))

            # 🛑 เงื่อนไขปิดงาน: ถ้า AI มั่นใจ ตอบเป็นข้อความดิบ และไม่สั่งเปิดเครื่องมือเพิ่มแล้ว
            if not current_tool_calls:
                return content or "Done"

            # 🛠️ สั่งรันเครื่องมือหลังบ้าน (Local Execution)
            for tc in current_tool_calls:
                observation = self.tools.dispatch(tc.name, tc.args)

                # ส่งผลลัพธ์กลับเข้าสู่ประวัติแชทเพื่อให้ AI อ่านในสเต็ปถัดไป
                self.history.append(Turn(
                    role="tool",
                    content=observation
                ))

        return "budget exhausted"

    def print_pretty_transcript(self):
        print("\n" + "="*80)
        print(f"TRANSCRIPT: LOCAL OLLAMA AGENT LOOP (Model: {self.model})")
        print("="*80)
        for i, turn in enumerate(self.history):
            prefix = f"[{i:02d} {turn.role.upper():>9}]"
            if turn.role == "user":
                print(f"{prefix} 👤 คุณถาม: {turn.content}")
            elif turn.role == "assistant":
                if turn.tool_calls:
                    for tc in turn.tool_calls:
                        print(f"{prefix} 🛠️ Ollama สั่งรันเครื่องมือ: '{tc.name}' พารามิเตอร์={tc.args}")
                if turn.content:
                    print(f"{prefix} 💬 Ollama ตอบกลับหน้าร้าน: {turn.content}")
            elif turn.role == "tool":
                print(f"{prefix} 💾 ผลลัพธ์จากเครื่องมือส่งคืน AI -> {turn.content}")
        print("="*80)

# ======================================================================
# 🚀 ทดสอบรันระบบจริงในเครื่องคอมพิวเตอร์ของคุณ
# ======================================================================
def main():
    # 1. ขึ้นทะเบียนเครื่องมือให้เอเจนต์ใช้งาน
    tools = ToolRegistry()
    tools.register(
        name="calculator",
        fn=calculator,
        description="ใช้สำหรับคำนวณสูตรคณิตศาสตร์ ตัวเลข บวกลบคูณหาร",
        parameters={
            "type": "object",
            "properties": {
                "expr": {"type": "string", "description": "สมการคณิตศาสตร์ เช่น '25000 * 0.05'"}
            },
            "required": ["expr"]
        }
    )

    # 2. เริ่มลูปการทำงาน
    # (หากเปลี่ยนไปใช้ llama3 ให้แก้ตรง model="llama3" นะครับ)
    agent = OllamaAgentLoop(tools=tools, model="qwen2.5")

    question = "ถ้าพี่น้องมีเงิน 25,432.75 บาท โดนหักภาษี ณ ที่จ่าย 5.3% จะได้เงินเหลือสุทธิกี่บาท? คิดให้ดูหน่อย"
    print(f"🤖 กำลังปลุกโมเดล {agent.model} ในเครื่องให้คิดเลขสักครู่ (ไม่ต้องใช้เน็ต)...")

    final_answer = agent.run(question)

    # 3. โชว์บันทึกกระบวนการทำงานคาจอ
    agent.print_pretty_transcript()
    print(f"\n✨ คำตอบสุดท้ายที่ได้จาก Ollama ในเครื่องคุณ: {final_answer}")

if __name__ == "__main__":
    main()

🤖 กำลังปลุกโมเดล qwen2.5 ในเครื่องให้คิดเลขสักครู่ (ไม่ต้องใช้เน็ต)...

TRANSCRIPT: LOCAL OLLAMA AGENT LOOP (Model: qwen2.5)
[00      USER] 👤 คุณถาม: ถ้าพี่น้องมีเงิน 25,432.75 บาท โดนหักภาษี ณ ที่จ่าย 5.3% จะได้เงินเหลือสุทธิกี่บาท? คิดให้ดูหน่อย
[01 ASSISTANT] 🛠️ Ollama สั่งรันเครื่องมือ: 'calculator' พารามิเตอร์={'expr': '25432.75 * (1 - 0.053)'}
[02      TOOL] 💾 ผลลัพธ์จากเครื่องมือส่งคืน AI -> 24084.81425
[03 ASSISTANT] 💬 Ollama ตอบกลับหน้าร้าน: หลังจากหักภาษี ณ ที่จ่าย 5.3% จากเงิน 25,432.75 บาท จะเหลือเงินสุทธิ约为24084.81泰铢。

✨ คำตอบสุดท้ายที่ได้จาก Ollama ในเครื่องคุณ: หลังจากหักภาษี ณ ที่จ่าย 5.3% จากเงิน 25,432.75 บาท จะเหลือเงินสุทธิ约为24084.81泰铢。


In [ ]:
from __future__ import annotations
import json
import requests
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable

@dataclass
class ToolCall:
    id: str
    name: str
    args: dict[str, Any]

@dataclass
class Turn:
    role: str       # 'system', 'user', 'assistant', 'tool'
    content: str | None = None
    tool_calls: list[ToolCall] = field(default_factory=list)
    tool_use_id: str | None = None

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}
        self._specs: dict[str, dict[str, Any]] = {}

    def register(self, name: str, fn: Callable[..., str], description: str, parameters: dict[str, Any]) -> None:
        self._tools[name] = fn
        self._specs[name] = {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": parameters
            }
        }

    def get_tools_spec(self) -> list[dict[str, Any]]:
        return list(self._specs.values())

    def dispatch(self, name: str, args: dict[str, Any]) -> str:
        fn = self._tools.get(name)
        if fn is None:
            return f"error: unknown tool {name!r}"
        try:
            return fn(**args)
        except Exception as e:
            return f"error: {e}"

# --- กล่องเครื่องมือคิดเลขแม่นยำ ---
def calculator(expr: str) -> str:
    try: return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e: return f"error: {e}"

# ======================================================================
# 🌟 คลาส AgentLoop เวอร์ชั่นคุมประพฤติภาษา (Strict Guardrails Agent)
# ======================================================================
class StrictThaiAgentLoop:
    def __init__(self, tools: ToolRegistry, model: str = "qwen2.5", max_turns: int = 10):
        self.ollama_url = "http://localhost:11434/api/chat"
        self.tools = tools
        self.model = model
        self.max_turns = max_turns
        self.history: list[Turn] = []

        # 🛡️ บรรจุสัญญากฎเหล็กตั้งแต่เริ่มเปิดระบบ
        self._init_system_prompt()

    def _init_system_prompt(self):
        # 🔥 ใส่กติกาบังคับตอบภาษาไทย ห้ามพ่นอักษรจีนเด็ดขาด
        strict_instructions = (
            "You are a helpful financial assistant. "
            "CRITICAL RULE: You must always respond to the user in fluent THAI language only. "
            "NEVER use Chinese characters or Chinese grammar under any circumstances. "
            "Keep the final answer professional and strictly in Thai."
        )
        self.history.append(Turn(role="system", content=strict_instructions))

    def run(self, user_message: str) -> str:
        self.history.append(Turn(role="user", content=user_message))

        for step in range(self.max_turns):
            api_messages = []
            # (ลูปแปลงประวัติเหมือนเดิม...)
            for turn in self.history:
                msg = {"role": turn.role}
                if turn.content is not None: msg["content"] = turn.content
                if turn.role == "assistant" and turn.tool_calls:
                    msg["tool_calls"] = [{"id": tc.id, "type": "function", "function": {"name": tc.name, "arguments": tc.args}} for tc in turn.tool_calls]
                if turn.role == "tool": msg["tool_call_id"] = turn.tool_use_id
                api_messages.append(msg)

            # 🌟 จุดเปลี่ยนอยู่ตรงนี้: ถ้าเครื่องมือรันครบแล้ว (รอบสรุปคำตอบสุดท้าย)
            # เราจะยัด JSON Schema บังคับผลลัพธ์ให้ Ollama พ่นตามสั่งเป๊ะๆ
            payload = {
                "model": self.model,
                "messages": api_messages,
                "stream": False,
                "tools": self.tools.get_tools_spec()
            }

            # 🔥 ถ้ารอบนี้คาดว่าจะเป็นรอบสรุปคำตอบ (เช่น รอบก่อนหน้าเพิ่งรัน TOOL เสร็จมา)
            # เราเปิดใช้ระบบ Structured Outputs บังคับรูปแบบผลลัพธ์เลย!
            if len(self.history) > 2 and self.history[-1].role == "tool":
                payload["format"] = {
                    "type": "object",
                    "properties": {
                        "final_amount": {"type": "number", "description": "เงินสุทธิที่เหลือ"},
                        "tax_paid": {"type": "number", "description": "ภาษีที่โดนหัก"}
                    },
                    "required": ["final_amount", "tax_paid"]
                }

            response = requests.post(self.ollama_url, json=payload)
            res_json = response.json()
            message = res_json.get("message", {})
            content = message.get("content", "")

            # --- ส่วนจัดการ Tool Call เหมือนเดิม ---
            api_tool_calls = message.get("tool_calls", [])
            current_tool_calls = []
            for tc in api_tool_calls:
                func = tc.get("function", {})
                tc_id = tc.get("id") or f"toolu_{uuid.uuid4().hex[:8]}"
                current_tool_calls.append(ToolCall(id=tc_id, name=func.get("name"), args=func.get("arguments", {})))

            self.history.append(Turn(role="assistant", content=content if content else None, tool_calls=current_tool_calls))

            if not current_tool_calls:
                # 🌟 ถ้าได้ผลลัพธ์เป็น JSON เราจับมาแกะแล้วจัดหน้าตาภาษาไทยเอง 100% หลังบ้านเลยครับ
                try:
                    data = json.loads(content)
                    return f"คำนวณสุทธิเรียบร้อยครับ: หักภาษี ณ ที่จ่ายไป {data['tax_paid']:.2f} บาท จะเหลือเงินสุทธิส่งถึงมือคุณทั้งหมด {data['final_amount']:.2f} บาทครับ"
                except:
                    return content or "Done"

            # รันเครื่องมือส่งค่ากลับตามปกติ
            for tc in current_tool_calls:
                observation = self.tools.dispatch(tc.name, tc.args)
                self.history.append(Turn(role="tool", content=observation, tool_use_id=tc.id))

        return "budget exhausted"

    def print_pretty_transcript(self):
        print("\n" + "="*80)
        print(f"TRANSCRIPT: STRICT THAI AGENT LOOP (Model: {self.model})")
        print("="*80)
        for i, turn in enumerate(self.history):
            prefix = f"[{i:02d} {turn.role.upper():>9}]"
            if turn.role == "system":
                print(f"{prefix} 🛡️ กฎระบบ: {turn.content}")
            elif turn.role == "user":
                print(f"{prefix} 👤 ถาม: {turn.content}")
            elif turn.role == "assistant":
                if turn.tool_calls:
                    for tc in turn.tool_calls:
                        print(f"{prefix} 🛠️ AI สั่งเปิด [ID: {tc.id}]: เครื่องมือ '{tc.name}' พารามิเตอร์={tc.args}")
                if turn.content:
                    print(f"{prefix} 💬 คำตอบหน้าร้าน: {turn.content}")
            elif turn.role == "tool":
                print(f"{prefix} 💾 Observation จากเครื่องมือกลับมา -> {turn.content}")
        print("="*80)

# ======================================================================
# 🚀 รันระบบทดสอบเวอร์ชั่นสมบูรณ์
# ======================================================================
def main():
    tools = ToolRegistry()
    tools.register(
        name="calculator",
        fn=calculator,
        description="ใช้สำหรับคำนวณสูตรคณิตศาสตร์ ตัวเลข บวกลบคูณหาร",
        parameters={
            "type": "object",
            "properties": {
                "expr": {"type": "string", "description": "สมการคณิตศาสตร์ เช่น '25432.75 * (1 - 0.053)'"}
            },
            "required": ["expr"]
        }
    )

    agent = StrictThaiAgentLoop(tools=tools, model="qwen2.5")
    question = "ถ้าพี่น้องมีเงิน 25,432.75 บาท โดนหักภาษี ณ ที่จ่าย 5.3% จะได้เงินเหลือสุทธิกี่บาท? คิดให้ดูหน่อย ตอบเป็นอังกิด"

    print("🤖 กำลังเปิดระบบเอเจนต์พร้อมควบคุมพฤติกรรมภาษา...")
    final_answer = agent.run(question)

    # โชว์ผลลัพธ์
    agent.print_pretty_transcript()
    print(f"\n✨ คำตอบสุดท้ายที่ได้ (ภาษาไทย 100%): {final_answer}")

if __name__ == "__main__":
    main()

🤖 กำลังเปิดระบบเอเจนต์พร้อมควบคุมพฤติกรรมภาษา...

TRANSCRIPT: STRICT THAI AGENT LOOP (Model: qwen2.5)
[00    SYSTEM] 🛡️ กฎระบบ: You are a helpful financial assistant. CRITICAL RULE: You must always respond to the user in fluent THAI language only. NEVER use Chinese characters or Chinese grammar under any circumstances. Keep the final answer professional and strictly in Thai.
[01      USER] 👤 ถาม: ถ้าพี่น้องมีเงิน 25,432.75 บาท โดนหักภาษี ณ ที่จ่าย 5.3% จะได้เงินเหลือสุทธิกี่บาท? คิดให้ดูหน่อย ตอบเป็นอังกิด
[02 ASSISTANT] 🛠️ AI สั่งเปิด [ID: call_eqnxcy3n]: เครื่องมือ 'calculator' พารามิเตอร์={'expr': '25432.75 * (1 - 0.053)'}
[03      TOOL] 💾 Observation จากเครื่องมือกลับมา -> 24084.81425
[04 ASSISTANT] 💬 คำตอบหน้าร้าน: {
    "final_amount": 24, "tax_paid": 16.95775
}
                		

✨ คำตอบสุดท้ายที่ได้ (ภาษาไทย 100%): คำนวณสุทธิเรียบร้อยครับ: หักภาษี ณ ที่จ่ายไป 16.96 บาท จะเหลือเงินสุทธิส่งถึงมือคุณทั้งหมด 24.00 บาทครับ


In [ ]:
import requests

def ask_ollama_directly(prompt: str, model: str = "qwen2.5"):
    url = "http://localhost:11434/api/generate" # ใช้ endpoint generate ธรรมดา
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    try:
        response = requests.post(url, json=payload)
        return response.json().get("response", "")
    except Exception as e:
        return f"🚨 ลืมรันเซลล์เปิด Ollama หรือเปล่าครับพี่น้อง? ({e})"

# ทดสอบถามตรงๆ
question = "ถ้าพี่น้องมีเงิน 25,432.75 บาท โดนหักภาษี ณ ที่จ่าย 5.3% จะได้เงินเหลือสุทธิกี่บาท? คิดให้ดูหน่อย"
print("👤 กำลังถาม LLM ตรงๆ (ไม่ผ่าน Agent)...")
print("-" * 50)
print(ask_ollama_directly(question))

👤 กำลังถาม LLM ตรงๆ (ไม่ผ่าน Agent)...
--------------------------------------------------
แน่นอนครับ/คะ นี่คือขั้นตอนในการคำนวณ：

1. หาจำนวนภาษีที่ต้องหัก : 
   นำเงิน总额25,432.75乘以税率5.3%（0.053），即：25,432.75 × 0.053 = 1,346.02.
   
2. คำนวณเงินสุทธิที่ได้ :
   เงิน总额减去所缴纳的税，即：25,432.75 - 1,346.02 = 24,086.73.

ดังนั้น, หลังจากหักภาษี ณ ที่จ่าย 5.3% คุณจะได้เงินสุทธิประมาณ 24,086.73 บาท.


In [ ]:
from __future__ import annotations
import json
import requests
import uuid  # 🌟 ใช้สร้าง ID แบบสุ่มเพื่อให้แน่ใจว่าไม่มีทางซ้ำกัน
from dataclasses import dataclass, field
from typing import Any, Callable

@dataclass
class ToolCall:
    id: str        # 🌟 ตัวประสาน ID (Correlator ID) ประจำแผลคำสั่งนี้
    name: str
    args: dict[str, Any]

@dataclass
class Turn:
    role: str       # 'user', 'assistant', 'tool'
    content: str | None = None

    # 🌟 ฝั่ง Assistant: สามารถสั่งรันเครื่องมือขนานกันได้หลายตัว (Parallel) พร้อม ID ของตัวเอง
    tool_calls: list[ToolCall] = field(default_factory=list)

    # 🌟 ฝั่ง Tool (Observation): ต้องประกาศตัวบอกว่า ผลลัพธ์นี้สืบเนื่องมาจาก ID คำสั่งไหน
    tool_use_id: str | None = None

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}
        self._specs: dict[str, dict[str, Any]] = {}

    def register(self, name: str, fn: Callable[..., str], description: str, parameters: dict[str, Any]) -> None:
        self._tools[name] = fn
        self._specs[name] = {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": parameters
            }
        }

    def get_tools_spec(self) -> list[dict[str, Any]]:
        return list(self._specs.values())

    def dispatch(self, name: str, args: dict[str, Any]) -> str:
        fn = self._tools.get(name)
        if fn is None:
            return f"error: unknown tool {name!r}"
        try:
            return fn(**args)
        except Exception as e:
            return f"error: {e}"

# --- ประกาศกล่องเครื่องมือ ---
def calculator(expr: str) -> str:
    try: return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e: return f"error: {e}"

def kv_get(key: str) -> str:
    # คลังข้อมูลจำลอง
    db = {"base_salary": "50000", "bonus_rate": "0.10"}
    return db.get(key, f"missing:{key}")

# ======================================================================
# 🌟 คลาส AgentLoop รูปแบบ Anthropic Schema (รองรับการสลับคิวผลลัพธ์)
# ======================================================================
class ParallelAgentLoop:
    def __init__(self, tools: ToolRegistry, model: str = "qwen2.5", max_turns: int = 10):
        self.ollama_url = "http://localhost:11434/api/chat"
        self.tools = tools
        self.model = model
        self.max_turns = max_turns
        self.history: list[Turn] = []

    def run(self, user_message: str) -> str:
        self.history.append(Turn(role="user", content=user_message))

        for step in range(self.max_turns):
            api_messages = []
            for turn in self.history:
                msg = {"role": turn.role}
                if turn.content is not None:
                    msg["content"] = turn.content

                if turn.role == "assistant" and turn.tool_calls:
                    msg["tool_calls"] = [
                        {
                            "id": tc.id,  # 🌟 ส่งโครงสร้าง ID ไปให้ประวัติของโมเดลรับรู้ด้วย
                            "type": "function",
                            "function": {"name": tc.name, "arguments": tc.args}
                        } for tc in turn.tool_calls
                    ]
                if turn.role == "tool":
                    # 🌟 จำลองการแมปไอดีขากลับข้ามค่าย (Anthropic จะใช้โครงสร้างเฉพาะตัว)
                    msg["tool_call_id"] = turn.tool_use_id

                api_messages.append(msg)

            payload = {
                "model": self.model,
                "messages": api_messages,
                "stream": False,
                "tools": self.tools.get_tools_spec()
            }

            try:
                response = requests.post(self.ollama_url, json=payload)
                res_json = response.json()
            except Exception as e:
                return f"🚨 เชื่อมต่อหลังบ้านล้มเหลว: {e}"

            message = res_json.get("message", {})
            content = message.get("content", "")
            api_tool_calls = message.get("tool_calls", [])

            current_tool_calls = []
            for tc in api_tool_calls:
                func = tc.get("function", {})
                # 🌟 ถ้า Ollama ไม่สร้าง ID มาให้ ระบบรันไทม์ของเราจะสุ่มรหัส ID สไตล์ Anthropic ให้ทันที
                tc_id = tc.get("id") or f"toolu_{uuid.uuid4().hex[:8]}"
                current_tool_calls.append(
                    ToolCall(id=tc_id, name=func.get("name"), args=func.get("arguments", {}))
                )

            self.history.append(Turn(
                role="assistant",
                content=content if content else None,
                tool_calls=current_tool_calls
            ))

            if not current_tool_calls:
                return content or "Done"

            # 🌟 จุดไคลแมกซ์: สั่งรันเครื่องมือขนาน และ แกล้งสลับลำดับผลลัพธ์ (Out of Order)
            # สมมติตัวอย่างว่า AI สั่งรัน 2 เครื่องมือพร้อมกัน รอบนี้เราจะเอาผลลัพธ์ตัวที่ 2 ส่งคืนเข้าแชทก่อนตัวแรก!
            batch_results = []
            for tc in current_tool_calls:
                observation = self.tools.dispatch(tc.name, tc.args)
                batch_results.append(Turn(role="tool", content=observation, tool_use_id=tc.id))

            # 🔥 แกล้งสลับคิวหลังบ้าน (ถ้าส่งคำสั่งพร้อมกัน 2 ตัว ให้สลับตัวท้ายขึ้นมานำหน้า)
            if len(batch_results) > 1:
                batch_results.reverse()
                print("⚠️ [ระบบเครือข่าย]: ตรวจพบการประมวลผลขนานแบบสลับคิว (Out of Order Execution!)")

            # บันทึกผลลัพธ์ที่สลับคิวแล้วลงไปในประวัติแชทจริง
            for result_turn in batch_results:
                self.history.append(result_turn)

        return "budget exhausted"

    def print_pretty_transcript(self):
        print("\n" + "="*80)
        print(f"TRANSCRIPT: ANTHROPIC SCHEMA (WITH TOOL_USE_ID)")
        print("="*80)
        for i, turn in enumerate(self.history):
            prefix = f"[{i:02d} {turn.role.upper():>9}]"
            if turn.role == "user":
                print(f"{prefix} 👤 ถาม: {turn.content}")
            elif turn.role == "assistant":
                if turn.tool_calls:
                    for tc in turn.tool_calls:
                        print(f"{prefix} 🛠️ AI สั่งเปิด [ID: {tc.id}]: เครื่องมือ '{tc.name}' พารามิเตอร์={tc.args}")
                if turn.content:
                    print(f"{prefix} 💬 คำตอบหน้าร้าน: {turn.content}")
            elif turn.role == "tool":
                print(f"{prefix} 💾 Observation คืนค่ากลับ [จาก ID: {turn.tool_use_id}] -> {turn.content}")
        print("="*80)

def main():
    tools = ToolRegistry()
    tools.register("calculator", calculator, "คิดเลข", {"type": "object", "properties": {"expr": {"type": "string"}}, "required": ["expr"]})
    tools.register("kv_get", kv_get, "ดึงข้อมูลจากคลัง", {"type": "object", "properties": {"key": {"type": "string"}}, "required": ["key"]})

    agent = ParallelAgentLoop(tools=tools, model="qwen2.5")
    # สั่งงานลุยโจทย์ที่ต้องเปิดเครื่องมือสองตัวขนานกัน
    final = agent.run("ช่วยไปดึงค่า 'base_salary' และค่า 'bonus_rate' จากคลังข้อมูลมาให้หน่อย")
    agent.print_pretty_transcript()

if __name__ == "__main__":
    main()

⚠️ [ระบบเครือข่าย]: ตรวจพบการประมวลผลขนานแบบสลับคิว (Out of Order Execution!)

TRANSCRIPT: ANTHROPIC SCHEMA (WITH TOOL_USE_ID)
[00      USER] 👤 ถาม: ช่วยไปดึงค่า 'base_salary' และค่า 'bonus_rate' จากคลังข้อมูลมาให้หน่อย
[01 ASSISTANT] 🛠️ AI สั่งเปิด [ID: call_b2ru5cnl]: เครื่องมือ 'kv_get' พารามิเตอร์={'key': 'base_salary'}
[01 ASSISTANT] 🛠️ AI สั่งเปิด [ID: call_6xirseox]: เครื่องมือ 'kv_get' พารามิเตอร์={'key': 'bonus_rate'}
[02      TOOL] 💾 Observation คืนค่ากลับ [จาก ID: call_6xirseox] -> 0.10
[03      TOOL] 💾 Observation คืนค่ากลับ [จาก ID: call_b2ru5cnl] -> 50000
[04 ASSISTANT] 💬 คำตอบหน้าร้าน: ได้ค่า 'base_salary' เป็น 50,000 และ 'bonus_rate' เป็น 10% จากคลังข้อมูลแล้วค่ะ.


In [ ]:
import requests
import json

# ======================================================================
# 1. TOOLS DEFINITION (ส่วนผสม: Tool Registry)
# ======================================================================
def calculator(expr: str) -> str:
    """เครื่องมือสำหรับคำนวณสมการตัวเลขคณิตศาสตร์"""
    try: return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e: return f"error: {e}"

# ผูกฟังก์ชันเข้ากับชื่อ เพื่อให้ระบบตรวจเรียกใช้ตามชื่อได้
AVAILABLE_TOOLS = {"calculator": calculator}

# ส่ง Specs ไปให้ Ollama รู้จักวิธีเรียกใช้งาน (Schema In)
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "ใช้คำนวณสูตรคณิตศาสตร์ ตัวเลข บวกลบคูณหาร",
            "parameters": {
                "type": "object",
                "properties": {"expr": {"type": "string", "description": "สมการคณิตศาสตร์ เช่น '25432.75 * 0.053'"}},
                "required": ["expr"]
            }
        }
    }
]

# ======================================================================
# 2. THE MINIMAL REACT AGENT LOOP
# ======================================================================
def run_minimal_agent(user_prompt: str, model: str = "qwen2.5", max_turns: int = 5):
    url = "http://localhost:11434/api/chat"

    # 🌟 ส่วนผสม: Message Buffer ที่โตขึ้นเรื่อยๆ
    messages = [
        {"role": "system", "content": "You are a precise assistant. Answer in THAI language only."},
        {"role": "user", "content": user_prompt}
    ]

    print(f"🎬 เริ่มต้นวิ่งลูป ReAct (จำกัดรอบสูงสุด Turn Budget = {max_turns} รอบ)")
    print("=" * 80)

    # 🌟 ส่วนผสม: Turn Budget (จำกัดรอบ loop ชัดเจน)
    for turn in range(max_turns):
        print(f"\n[TURN {turn+1}/{max_turns}] 🧠 กำลังส่งให้ LLM คิดและตัดสินใจ...")

        payload = {"model": model, "messages": messages, "stream": False, "tools": TOOLS_SCHEMA}
        response = requests.post(url, json=payload).json()

        assistant_message = response.get("message", {})
        messages.append(assistant_message) # บันทึกคำตอบของ AI ลง buffer

        tool_calls = assistant_message.get("tool_calls", [])

        # 🌟 ส่วนผสม: Stop Condition (ถ้าไม่มีการเรียก Tool แสดงว่างานเสร็จสิ้น)
        if not tool_calls:
            print("🛑 [STOP CONDITION] -> AI ไม่สั่งเรียกเครื่องมือเพิ่มแล้ว แปลว่าจบภารกิจ!")
            return assistant_message.get("content", "")

        # 🌟 ส่วนผสม: Execution & Observation Formatter (รันเครื่องมือและสมานแผลแปลงเป็น String)
        for tool_call in tool_calls:
            tool_name = tool_call["function"]["name"]
            tool_args = tool_call["function"]["arguments"]
            tool_id = tool_call.get("id", "call_1")

            print(f"🛠️  AI สั่ง Action -> เปิดใช้เครื่องมือ '{tool_name}' คอยจิ้มค่า: {tool_args}")

            # รันระบบจริงหลังบ้าน
            if tool_name in AVAILABLE_TOOLS:
                result_string = AVAILABLE_TOOLS[tool_name](**tool_args)
            else:
                result_string = f"error: tool {tool_name} not found"

            print(f"💾 Observation -> ผลลัพธ์ที่ได้กลับมา: {result_string}")

            # ยัดผลลัพธ์ที่แปลงเป็นข้อความธรรมดากลับเข้าคิวประวัติแชท (Message Buffer)
            messages.append({
                "role": "tool",
                "content": result_string,
                "tool_call_id": tool_id
            })

    # 🌟 ส่วนผสม: Stop Condition (กรณีงบประมาณรอบหมดก่อนงานเสร็จ)
    print("\n🚨 [STOP CONDITION] -> Turn budget exhausted! ตัดจบเพื่อป้องกันลูปนรก")
    return "ไม่สามารถหาคำตอบได้ภายในจำนวนรอบที่กำหนด"

# ======================================================================
# 🚀 ทดสอบรันระบบจริงกลางสนาม Colab
# ======================================================================
prompt_test = "ถ้าพี่น้องมีเงิน 25,432.75 บาท โดนหักภาษี ณ ที่จ่าย 5.3% จะได้เงินเหลือสุทธิกี่บาท? คิดให้ดูหน่อย"
final_answer = run_minimal_agent(prompt_test)

print("\n" + "="*80)
print(f"✨ คำตอบสุดท้ายหน้าร้าน:\n{final_answer}")
print("="*80)

🎬 เริ่มต้นวิ่งลูป ReAct (จำกัดรอบสูงสุด Turn Budget = 5 รอบ)

[TURN 1/5] 🧠 กำลังส่งให้ LLM คิดและตัดสินใจ...
🛠️  AI สั่ง Action -> เปิดใช้เครื่องมือ 'calculator' คอยจิ้มค่า: {'expr': '25432.75 * (1 - 0.053)'}
💾 Observation -> ผลลัพธ์ที่ได้กลับมา: 24084.81425

[TURN 2/5] 🧠 กำลังส่งให้ LLM คิดและตัดสินใจ...
🛑 [STOP CONDITION] -> AI ไม่สั่งเรียกเครื่องมือเพิ่มแล้ว แปลว่าจบภารกิจ!

✨ คำตอบสุดท้ายหน้าร้าน:
พี่น้องจะได้รับเงินเหลือสุทธิ约为24,084.81泰铢（四舍五入至两位小数）。
